# 🤖 Basic LangGraph Agent Setup Tutorial

## Welcome to Agentic AI Development!

This comprehensive notebook will guide you through building your first AI agent using **LangGraph** - a powerful framework for creating stateful, multi-step AI applications. By the end of this tutorial, you'll have a fully functional AI agent that can engage in conversations, maintain context, and handle various types of queries.

### 🎯 Learning Objectives

After completing this tutorial, you will be able to:

1. **🔧 Set up the environment** - Install and configure all necessary libraries for LangGraph development
2. **🏗️ Build basic agents** - Create simple conversational AI agents using StateGraph
3. **🧠 Manage agent state** - Understand and implement state management for conversation memory
4. **🔄 Design agent workflows** - Create multi-node graphs with conditional logic
5. **🛠️ Handle errors gracefully** - Implement robust error handling and input validation
6. **🧪 Test and debug** - Use comprehensive testing strategies for agent validation
7. **🚀 Explore real applications** - See practical use cases and deployment considerations

### 🌟 What Makes This Tutorial Special?

- **Hands-on approach**: Every concept is immediately followed by working code examples
- **Production-ready**: Includes error handling, validation, and best practices
- **Comprehensive**: Covers everything from basic setup to advanced features
- **Interactive**: Includes testing playground for experimenting with your own queries
- **Well-documented**: Detailed explanations and comments throughout

### 📋 Prerequisites

- **Python 3.8+** (We'll verify this during setup)
- **Basic Python knowledge** (functions, classes, imports)
- **Azure OpenAI access** (API key and endpoint)
- **Jupyter Notebook environment** (JupyterLab, VS Code, or Jupyter Notebook)

### 🗺️ Tutorial Roadmap

1. **Environment Setup** - Install required packages and verify configuration
2. **Core Concepts** - Learn LangGraph fundamentals and state management
3. **Basic Agent** - Build your first conversational AI agent
4. **Enhanced Features** - Add input validation and error handling
5. **Testing & Validation** - Comprehensive testing strategies
6. **Real-world Applications** - Explore various use cases and scenarios
7. **Troubleshooting** - Common issues and how to resolve them
8. **Next Steps** - Advanced features and deployment considerations

### 💡 Key Technologies We'll Use

- **🦜 LangGraph**: Framework for building stateful AI applications
- **🔗 LangChain**: Foundation for LLM applications and integrations
- **🧠 Azure OpenAI**: Powerful language models (GPT-4, GPT-3.5-turbo)
- **📊 Python**: Core programming language for AI development
- **🔄 State Management**: Maintaining conversation context and memory

### 🚀 Ready to Build Your First AI Agent?

Let's start this exciting journey into agentic AI development! Each section builds upon the previous one, so we recommend following the tutorial in order. Don't worry if you're new to LangGraph - we'll explain everything step by step.

---

*💡 **Tip**: Feel free to experiment with the code as we go along. The best way to learn is by doing!*

## 2. Installation and Setup {#setup}

### 📋 Prerequisites Checklist

Before we begin, ensure you have:
- ✅ **Python 3.8+** (Check with `python --version`)
- ✅ **Jupyter Notebook** or **JupyterLab** 
- ✅ **OpenAI API Key** (Get one from [OpenAI Platform](https://platform.openai.com/))
- ✅ **Stable Internet Connection** for package downloads

### 🌍 Setting up Virtual Environment (Recommended)

```bash
# Create a new virtual environment
python -m venv rag_agent_env

# Activate the environment
# Windows:
rag_agent_env\Scripts\activate
# macOS/Linux:
source rag_agent_env/bin/activate

# Install Jupyter if not already available
pip install jupyter notebook ipywidgets
```

### 📦 Required Libraries Overview

We'll be installing several key libraries:
- **langgraph**: Core framework for building our agent
- **langchain**: Foundation for LLM interactions
- **openai**: OpenAI API integration
- **faiss-cpu**: Vector store for document indexing
- **sentence-transformers**: Text embeddings
- **pypdf**: PDF document processing
- **tiktoken**: Token counting for OpenAI models

In [ ]:
# Install required packages
# %pip install --quiet pydantic
# %pip install --quiet langgraph langchain langchain-openai openai
# %pip install --quiet langchain-community python-dotenv

print("✅ All packages installed successfully!")

### Step 2: Import Required Libraries
Now let's import all the necessary libraries for our LangGraph agent.

In [ ]:
# Import required libraries for LangGraph agent
import os
import sys
from typing import TypedDict, List, Dict, Any
from dotenv import load_dotenv

# LangGraph and LangChain imports
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

print("✅ All libraries imported successfully!")

### 3. Azure OpenAI Chat Model Integration (#config)

This example demonstrates how to use the `AzureChatOpenAI` class from the `langchain_openai` package to interact with Azure OpenAI's GPT models.

#### Key Steps:
1. **Setup**: Import necessary libraries and retrieve credentials for the Azure OpenAI API.
2. **Credentials**: Ensure the Azure API key and endpoint are correctly set.
3. **Error Handling**: Check if the API key and endpoint are missing, and raise errors if needed.
4. **Chat Model**: Initialize the `AzureChatOpenAI` model by providing required credentials like API key, endpoint, model type, and API version.

#### Notes:
- Ensure that Azure OpenAI is set up and environment variables for the API key and endpoint are configured.


In [ ]:
sys.path.insert(1, '../')

In [ ]:
import init_creds as creds

AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
    )

print("✅ LLM and Embeddings initialized successfully!")
print(f"📝 Language Model: {llm.model_name}")
print(f"🔍 Embeddings Model: text-embedding-3-small")

## 4. Basic LangGraph Agent Setup {#basic-agent}

### Understanding LangGraph State
LangGraph uses a state-based approach where each node in the graph can read and modify a shared state. Let's define our state structure.

In [ ]:
# Define the state structure for our agent
class AgentState(TypedDict):
    """
    State schema for our LangGraph agent.
    This defines the data structure that flows through our graph nodes.
    """
    messages: List[BaseMessage]  # Store conversation history
    user_input: str             # Current user input
    agent_response: str         # Agent's response
    step_count: int            # Track number of steps

print("✅ Agent state schema defined!")
print("State includes: messages, user_input, agent_response, step_count")

### Step 1: Create Agent Node Functions
Let's create the core functions that will serve as nodes in our LangGraph.

In [ ]:
def llm_node(state: AgentState) -> AgentState:
    """
    Main LLM node that processes user input and generates responses.
    This is where the actual LLM call happens.
    """
    print(f"🤖 Processing user input: '{state['user_input']}'")
    
    # Create system message to set the agent's behavior
    system_message = SystemMessage(content="""
    You are a helpful AI assistant. Provide clear, concise, and helpful responses.
    Be friendly and engaging while maintaining professionalism.
    """)
    
    # Create human message from user input
    human_message = HumanMessage(content=state["user_input"])
    
    # Prepare messages for LLM (include conversation history + new message)
    messages = [system_message] + state.get("messages", []) + [human_message]
    
    try:
        # Make the LLM call
        response = llm.invoke(messages)
        
        # Extract the response content
        agent_response = response.content
        
        # Update state with the new response
        updated_messages = state.get("messages", []) + [human_message, AIMessage(content=agent_response)]
        
        return {
            **state,
            "messages": updated_messages,
            "agent_response": agent_response,
            "step_count": state.get("step_count", 0) + 1
        }
    
    except Exception as e:
        error_message = f"Error calling LLM: {str(e)}"
        print(f"❌ {error_message}")
        
        return {
            **state,
            "agent_response": error_message,
            "step_count": state.get("step_count", 0) + 1
        }

print("✅ LLM node function created!")

### Step 2: Build the LangGraph
Now let's create the actual graph structure using our defined node.

In [ ]:
# Create the StateGraph
def create_agent_graph():
    """
    Creates and returns a LangGraph StateGraph with a single LLM node.
    """
    # Initialize the StateGraph with our AgentState schema
    workflow = StateGraph(AgentState)
    
    # Add the LLM node to the graph
    workflow.add_node("llm", llm_node)
    
    # Set the entry point (where the graph starts)
    workflow.set_entry_point("llm")
    
    # Define the flow: LLM node -> END
    workflow.add_edge("llm", END)
    
    # Compile the graph
    graph = workflow.compile()
    
    return graph

# Create the agent graph
agent_graph = create_agent_graph()

print("✅ LangGraph agent created successfully!")
print("Graph structure: START -> LLM Node -> END")

## 5. Testing and Examples {#testing}

### Step 1: Helper Function for Easy Testing
Let's create a helper function to make it easy to interact with our agent.

In [ ]:
def chat_with_agent(user_input: str, conversation_history: List[BaseMessage] = None) -> Dict[str, Any]:
    """
    Helper function to interact with the agent easily.
    
    Args:
        user_input: The user's message/question
        conversation_history: Optional previous conversation messages
    
    Returns:
        Dictionary with agent response and updated state
    """
    # Prepare initial state
    initial_state = {
        "user_input": user_input,
        "messages": conversation_history or [],
        "agent_response": "",
        "step_count": 0
    }
    
    # Run the agent graph
    final_state = agent_graph.invoke(initial_state)
    
    return {
        "response": final_state["agent_response"],
        "step_count": final_state["step_count"],
        "conversation_history": final_state["messages"]
    }

def display_response(user_input: str, result: Dict[str, Any]):
    """Helper function to display conversation in a nice format."""
    print("=" * 50)
    print(f"👤 User: {user_input}")
    print(f"🤖 Agent: {result['response']}")
    print(f"📊 Steps taken: {result['step_count']}")
    print("=" * 50)

print("✅ Helper functions created!")

### Step 2: Test the Agent with Simple Examples

In [ ]:
# Test 1: Simple greeting
print("🧪 Test 1: Simple Greeting")
user_message = "Hello! How are you today?"
result = chat_with_agent(user_message)
display_response(user_message, result)

In [ ]:
# Test 2: Technical question
print("\n🧪 Test 2: Technical Question")
user_message = "Explain what Python is in simple terms."
result = chat_with_agent(user_message)
display_response(user_message, result)

### Step 3: Test Conversation Memory
Let's test if our agent can maintain conversation context.

In [ ]:
# Test 3: Conversation with memory
print("🧪 Test 3: Conversation Memory")

# First message
conversation_history = []
user_message1 = "My name is John and I'm learning Python."
result1 = chat_with_agent(user_message1, conversation_history)
display_response(user_message1, result1)

# Second message using the conversation history
user_message2 = "What did I tell you my name was?"
result2 = chat_with_agent(user_message2, result1["conversation_history"])
display_response(user_message2, result2)

## 6. Advanced Agent with Additional Features {#advanced-agent}

### Adding Input Validation and Error Handling
Let's enhance our agent with better error handling and input validation.

In [ ]:
# Enhanced agent with input validation and preprocessing
def validate_input_node(state: AgentState) -> AgentState:
    """
    Validates and preprocesses user input before sending to LLM.
    """
    user_input = state["user_input"].strip()
    
    # Check for empty input
    if not user_input:
        return {
            **state,
            "agent_response": "Please provide a valid input. I'm here to help!",
            "step_count": state.get("step_count", 0) + 1
        }
    
    # Check for inappropriate content (basic filter)
    inappropriate_words = ["spam", "test123", "asdfgh"]  # Add more as needed
    if any(word in user_input.lower() for word in inappropriate_words):
        return {
            **state,
            "agent_response": "I notice your message might not be appropriate. Please rephrase your question.",
            "step_count": state.get("step_count", 0) + 1
        }
    
    # Input is valid, continue to LLM
    return state

def enhanced_llm_node(state: AgentState) -> AgentState:
    """
    Enhanced LLM node with better error handling and response formatting.
    """
    print(f"🤖 Processing validated input: '{state['user_input']}'")
    
    # Create system message with enhanced instructions
    system_message = SystemMessage(content="""
    You are a helpful, knowledgeable, and friendly AI assistant. 
    Guidelines:
    - Provide clear, accurate, and helpful responses
    - If you're uncertain about something, acknowledge it
    - Keep responses concise but informative
    - Use examples when helpful
    - Be encouraging and supportive
    """)
    
    human_message = HumanMessage(content=state["user_input"])
    messages = [system_message] + state.get("messages", []) + [human_message]
    
    try:
        # Make the LLM call with retry logic
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = llm.invoke(messages)
                agent_response = response.content
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    raise e
                print(f"⚠️ Attempt {attempt + 1} failed, retrying...")
        
        # Update state
        updated_messages = state.get("messages", []) + [human_message, AIMessage(content=agent_response)]
        
        return {
            **state,
            "messages": updated_messages,
            "agent_response": agent_response,
            "step_count": state.get("step_count", 0) + 1
        }
    
    except Exception as e:
        error_message = f"I apologize, but I'm experiencing technical difficulties. Please try again later. (Error: {str(e)})"
        return {
            **state,
            "agent_response": error_message,
            "step_count": state.get("step_count", 0) + 1
        }

print("✅ Enhanced agent nodes created!")

In [ ]:
# Create enhanced agent graph
def create_enhanced_agent_graph():
    """
    Creates an enhanced LangGraph with input validation and better error handling.
    """
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("validate", validate_input_node)
    workflow.add_node("llm", enhanced_llm_node)
    
    # Set entry point
    workflow.set_entry_point("validate")
    
    # Define conditional flow
    def should_continue_to_llm(state: AgentState) -> str:
        """Decide whether to continue to LLM or end based on validation."""
        if "Please provide a valid input" in state.get("agent_response", "") or \
           "might not be appropriate" in state.get("agent_response", ""):
            return END
        return "llm"
    
    # Add conditional edges
    workflow.add_conditional_edges(
        "validate",
        should_continue_to_llm,
        {
            "llm": "llm",
            END: END
        }
    )
    
    workflow.add_edge("llm", END)
    
    return workflow.compile()

# Create the enhanced agent
enhanced_agent = create_enhanced_agent_graph()

print("✅ Enhanced LangGraph agent created!")
print("Graph structure: START -> Validate -> [LLM | END] -> END")

## 7. Use Cases and Applications {#use-cases}

### Common Use Cases for LangGraph Agents

LangGraph agents can be used in various scenarios:

In [ ]:
# Example use cases demonstration
use_cases = {
    "Customer Support": "How do I reset my password?",
    "Educational Assistant": "Explain the concept of machine learning to a beginner.",
    "Code Helper": "Write a Python function to calculate factorial.",
    "Creative Writing": "Write a short story about a robot learning to cook.",
    "Data Analysis": "What are the key steps in exploratory data analysis?"
}

print("🎯 Demonstrating different use cases for our LangGraph agent:\n")

for use_case, example_query in use_cases.items():
    print(f"📋 Use Case: {use_case}")
    print(f"📝 Example Query: '{example_query}'")
    
    # Test with enhanced agent
    result = enhanced_agent.invoke({
        "user_input": example_query,
        "messages": [],
        "agent_response": "",
        "step_count": 0
    })
    
    print(f"🤖 Agent Response: {result['agent_response'][:150]}...")
    print("-" * 80)
    print()

## 8. Troubleshooting Common Issues {#troubleshooting}

### Common Problems and Solutions

Here are the most common issues you might encounter and how to fix them:

In [ ]:
# Troubleshooting function
def run_diagnostics():
    """
    Run diagnostic tests to check if everything is working correctly.
    """
    print("🔧 Running LangGraph Agent Diagnostics...\n")
    
    # Test 1: Check imports
    try:
        from langgraph.graph import StateGraph, END
        from langchain_openai import ChatOpenAI
        print("✅ Import test: All required libraries imported successfully")
    except ImportError as e:
        print(f"❌ Import test failed: {e}")
        print("Solution: Run '%pip install --upgrade langgraph langchain langchain-openai'")
        return False
    
    # Test 2: Check API key
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key and api_key.startswith("sk-"):
        print("✅ API Key test: OpenAI API key is configured")
    else:
        print("❌ API Key test: OpenAI API key not found or invalid")
        print("Solution: Set your OpenAI API key in environment variables")
        return False
    
    # Test 3: Check LLM connection
    try:
        test_llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=10)
        test_response = test_llm.invoke([HumanMessage(content="Hi")])
        print("✅ LLM Connection test: Successfully connected to OpenAI")
    except Exception as e:
        print(f"❌ LLM Connection test failed: {e}")
        print("Solution: Check your API key and internet connection")
        return False
    
    # Test 4: Check agent creation
    try:
        test_graph = create_agent_graph()
        print("✅ Agent Creation test: LangGraph agent created successfully")
    except Exception as e:
        print(f"❌ Agent Creation test failed: {e}")
        return False
    
    # Test 5: End-to-end test
    try:
        test_result = chat_with_agent("Hello, this is a test.")
        if "response" in test_result and test_result["response"]:
            print("✅ End-to-end test: Agent responded successfully")
        else:
            print("❌ End-to-end test: Agent didn't respond properly")
            return False
    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        return False
    
    print("\n🎉 All diagnostics passed! Your LangGraph agent is working correctly.")
    return True

# Common troubleshooting tips
troubleshooting_tips = {
    "ImportError: cannot import name 'Graph' from 'langgraph'": [
        "Update langgraph: pip install --upgrade langgraph",
        "Use StateGraph instead of Graph: from langgraph.graph import StateGraph",
        "Check Python version (3.8+ required)"
    ],
    "OpenAI API Error": [
        "Verify API key is correct and active",
        "Check you have sufficient API credits",
        "Ensure internet connection is stable",
        "Try different model (gpt-3.5-turbo vs gpt-4)"
    ],
    "Memory/Context Issues": [
        "Check conversation_history is passed correctly",
        "Verify state structure matches AgentState TypedDict",
        "Ensure messages are BaseMessage objects"
    ],
    "Performance Issues": [
        "Reduce max_tokens for faster responses",
        "Use gpt-3.5-turbo instead of gpt-4 for speed",
        "Implement caching for repeated queries"
    ]
}

print("📚 Troubleshooting Guide:")
print("=" * 50)
for issue, solutions in troubleshooting_tips.items():
    print(f"\n❓ Issue: {issue}")
    for i, solution in enumerate(solutions, 1):
        print(f"   {i}. {solution}")

print("\n" + "=" * 50)

In [ ]:
# Run the diagnostic test
print("🚀 Running comprehensive diagnostics...")
run_diagnostics()

## 9. Interactive Testing Playground

### Try Your Own Questions!
Use the cell below to test your agent with custom inputs.

In [ ]:
# Interactive testing - Modify the question below to test your agent!
your_question = "What is the difference between machine learning and artificial intelligence?"

print("🎮 Interactive Agent Testing")
print("=" * 50)
print(f"Your question: {your_question}")
print("=" * 50)

# Test with the enhanced agent
result = enhanced_agent.invoke({
    "user_input": your_question,
    "messages": [],
    "agent_response": "",
    "step_count": 0
})

print(f"🤖 Agent Response:\n{result['agent_response']}")
print(f"\n📊 Processing steps: {result['step_count']}")

## 10. Conclusion and Next Steps

### 🎉 Congratulations!
You have successfully built a complete LangGraph AI agent! Here's what you've accomplished:

#### ✅ What You've Learned:
1. **Environment Setup**: Installed and configured all necessary libraries
2. **LangGraph Basics**: Understanding of StateGraph, nodes, and edges  
3. **LLM Integration**: Connected OpenAI's GPT models to your agent
4. **State Management**: Managed conversation history and agent state
5. **Error Handling**: Implemented robust error handling and validation
6. **Testing**: Created comprehensive tests and diagnostics
7. **Use Cases**: Explored various applications for AI agents

#### 🚀 Next Steps and Advanced Features:
1. **Multi-Node Agents**: Add more specialized nodes (e.g., web search, database queries)
2. **Tool Integration**: Connect external APIs and tools
3. **Memory Systems**: Implement persistent memory across sessions
4. **Custom Prompting**: Advanced prompt engineering techniques
5. **Deployment**: Deploy your agent to production environments
6. **Monitoring**: Add logging and performance monitoring

#### 📚 Additional Resources:
- [LangGraph Documentation](https://python.langchain.com/docs/langgraph)
- [LangChain Community](https://github.com/langchain-ai/langchain)
- [OpenAI API Documentation](https://platform.openai.com/docs)

### 💡 Final Tips:
- Always test your agent thoroughly before deployment
- Monitor API usage and costs
- Keep your API keys secure
- Stay updated with library versions
- Join the LangChain community for support and updates

**Happy coding with AI agents! 🤖✨**